In [1]:
import pandas as pd
import re
import os

train_df = pd.read_csv("../data/amazon_polarity_train.csv")
print(train_df.shape)

(3600000, 3)


In [2]:
train_subset = train_df.sample(n=50000, random_state=42).reset_index(drop=True)
print(train_subset.shape)
print(train_subset["label"].value_counts())

(50000, 3)
label
1    25039
0    24961
Name: count, dtype: int64


In [3]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

train_subset["clean_content"] = train_subset["content"].apply(clean_text)
print(train_subset[["content", "clean_content"]].head(3))

                                             content  \
0  This product consists of a piece of thin flexi...   
1  Even on the lowest setting, the toast is too d...   
2  I enjoyed this disc. The video is stunning. I ...   

                                       clean_content  
0  This product consists of a piece of thin flexi...  
1  Even on the lowest setting, the toast is too d...  
2  I enjoyed this disc. The video is stunning. I ...  


In [4]:
before = len(train_subset)
train_subset = train_subset[train_subset["clean_content"].str.len() >= 10].reset_index(drop=True)
after = len(train_subset)
print(f"Dropped {before - after} rows that were empty/too short")
print(train_subset.shape)

Dropped 0 rows that were empty/too short
(50000, 4)


In [5]:
before = len(train_subset)
train_subset = train_subset.drop_duplicates(subset="clean_content").reset_index(drop=True)
after = len(train_subset)
print(f"Dropped {before - after} duplicate rows")
print(train_subset.shape)

Dropped 3 duplicate rows
(49997, 4)


## Day 3 Decision: Label Noise
Some reviews identified in Day 2 appear ambiguously or incorrectly labeled
(e.g., sarcastic tone). Given dataset scale, these are treated as irreducible
label noise rather than manually corrected — a documented limitation rather
than a blocker.

In [6]:
from sklearn.model_selection import train_test_split

train_val, test = train_test_split(
    train_subset, test_size=0.15, stratify=train_subset["label"], random_state=42
)

train, val = train_test_split(
    train_val, test_size=0.1765, stratify=train_val["label"], random_state=42
)

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (34996, 4)
Validation: (7501, 4)
Test: (7500, 4)


In [7]:
print("Train balance:\n", train["label"].value_counts(normalize=True))
print("\nVal balance:\n", val["label"].value_counts(normalize=True))
print("\nTest balance:\n", test["label"].value_counts(normalize=True))

Train balance:
 label
1    0.5008
0    0.4992
Name: proportion, dtype: float64

Val balance:
 label
1    0.500733
0    0.499267
Name: proportion, dtype: float64

Test balance:
 label
1    0.5008
0    0.4992
Name: proportion, dtype: float64


In [8]:
os.makedirs("../data/splits", exist_ok=True)

train.to_csv("../data/splits/train.csv", index=False)
val.to_csv("../data/splits/val.csv", index=False)
test.to_csv("../data/splits/test.csv", index=False)

print("Saved all three splits as CSV.")

Saved all three splits as CSV.


In [9]:
check_train = pd.read_csv("../data/splits/train.csv")
print(check_train.shape)
print(check_train.head(2))

(34996, 4)
   label              title  \
0      1  cute family movie   
1      0          Good luck   

                                             content  \
0  BUG OFF....ME AND MY DAUGHTER WATCH THIS TOGET...   
1  Mine didn't come with the digital copy. Amzon ...   

                                       clean_content  
0  BUG OFF....ME AND MY DAUGHTER WATCH THIS TOGET...  
1  Mine didn't come with the digital copy. Amzon ...  
